In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.animation as animation
import skimage.measure
from pathlib import Path
from numpy.typing import NDArray
from matplotlib.image import AxesImage as Artist

def load_image(path: Path, reduction_box: tuple[int,int] = (5, 5)) -> NDArray[np.floating]:
    """Load an image and apply some initial pre-processing operations. There are
    assumpting that go into this. """ 

    img = mpimg.imread(path)

    assert img.shape[-1] >= 3, f"Was expecting atleast three channels, go {img.shape}"

    # Collapse the three-channels into a single channel
    img = np.dot(img[...,:3], [0.2989, 0.5870, 0.1140])
    
    # Reorder the axis and reduce the image size to avoid access compute
    img = np.swapaxes(img, 0, 1)
    img = skimage.measure.block_reduce(img, reduction_box, np.max)

    return img


In [2]:
image = load_image(path=Path("20230915_112407.jpg"))

print(f"Loaded image shape: {image.shape}")

Loaded image shape: (807, 454)


In [20]:

def plot_3panel(
    axes: tuple[plt.Axes, plt.Axes, plt.Axes],
    img: NDArray[np.floating], 
    ft_img: NDArray[np.complex128],
    title_str: str | None = None,
) -> tuple[Artist, ...]:

    fft_freqs_0 = np.fft.fftshift(np.fft.fftfreq(img.shape[0]) * 2*np.pi)
    fft_freqs_1 = np.fft.fftshift(np.fft.fftfreq(img.shape[1]) * 2.*np.pi)
    extent = (
        np.min(fft_freqs_1),
        np.max(fft_freqs_1),
        np.min(fft_freqs_0),
        np.max(fft_freqs_0),
    )
    
    ax1, ax2, ax3 = axes
    
    ax1_artist = ax1.imshow(
        img
    )
    
    ax2_artist = ax2.imshow(
        np.log10(np.abs(ft_img)**2.),
        cmap='bwr',
        extent=extent
    )
    
    ax3_artist = ax3.imshow(
        np.angle(ft_img),
        cmap='bwr',
        extent=extent
    )
    
    artists = (ax1_artist, ax2_artist, ax3_artist)
    
    if title_str:
        ax2_title = ax2.text(
            0.0, 1.05, title_str, transform=ax2.transAxes
        )
        artists = artists + (ax2_title, )
    
    return artists
    
    


In [ ]:

def make_cut_inner(
    image: NDArray[np.floating], 
    radius: float, 
    axes: tuple[plt.Axes, ...] | None = None,
    mode: str = "inner"
) -> tuple[Artist, ...]:
    
    assert mode in ("inner", "outer"), f"Not an expected mode, inner or outer"
    
    if axes is None:
        fig, axes = plt.subplots(1, 3)
        
    y = np.arange(image.shape[0])
    x = np.arange(image.shape[1])

    xx, yy = np.meshgrid(x, y)

    center = np.array(image.shape) // 2

    mask = np.sqrt(
        (xx - center[1])**2 + (yy - center[0])**2
    ) < radius

    if mode == "outer":
        mask = ~mask

    ft_img = np.fft.fftshift(
        np.fft.fft2(image)
    )
    ft_img[mask] = 0+0j
    mask_img = np.fft.ifft2(np.fft.fftshift(ft_img))

    title = f"{mode} clip {radius}"

    return plot_3panel(axes=axes, img=mask_img.real, ft_img=ft_img, title_str=title)

    
plt.close("all")


def make_animation(
    mode: str, out_path: Path, image: NDArray[np.floating], max_scale: int=200
) -> Path:
    fig, axes = plt.subplots(1,3)

    frames = []

    for radius in range(max_scale):
        if radius % 20 == 0:
            print(f"{radius=}")
        frames.append(
            make_cut_inner(axes=axes, image=image, radius=radius, mode=mode)
        )

    print(f"Writing {out_path=}")
    inner_clip_animation = animation.ArtistAnimation(fig=fig, artists=frames, blit=True, interval=50)
    inner_clip_animation.save(out_path, writer="ffmpeg", fps=15)
    print(f"Finished writing {out_path=}")

make_animation(mode="inner", out_path=Path("inner_clip.mp4"), image=image, max_scale=400)
make_animation(mode="outer", out_path=Path("outer_clip.mp4"), image=image, max_scale=400)




radius=0


/var/folders/3b/kbsb64bs78d1qzgmpfkgwq4800_fyl/T/ipykernel_65178/1188449909.py:24: RuntimeWarning: divide by zero encountered in log10
  np.log10(np.abs(ft_img)**2.),


radius=20
radius=40
radius=60
radius=80
radius=100
radius=120
radius=140
radius=160
radius=180
radius=200
radius=220
radius=240
radius=260
radius=280
radius=300
radius=320
radius=340
radius=360
radius=380
Writing out_path=PosixPath('inner_clip.mp4')
